# Trovare i collegamenti tra tabelle SQL

Questo notebook legge tutti i file `.sql` di una cartella, estrae tabelle e colonne, mostra **com'è collegato il database prima dell'analisi** (chiavi esterne già dichiarate e **gruppi di tabelle** con nomi simili) e cerca i **possibili collegamenti** (chiavi esterne mancanti) in base a:

1. **colonne con lo stesso nome** in più tabelle
2. **colonne con nomi simili** (`COD_ART` / `CODART` / `CodArticolo`)
3. **nomi che rimandano a una tabella** (`id_articolo` → tabella `articoli`) e chiavi primarie
4. **tabelle con chiave primaria composta**: per ognuna, a cosa si collega ogni colonna della chiave
5. **qualità dello schema**: integrità, tipo di relazione (1:1, 1:N, N:M), normalizzazione e relazioni ridondanti
6. **diagnostica**: tutti i problemi come errori e avvisi di un compilatore, con file e riga e la regola di progettazione violata
7. **ontologia**: OWL e mapping R2RML generati dallo schema
8. **modello relazionale**: lo schema logico, esportabile in testo, Markdown, Mermaid, PlantUML, Graphviz, DBML e JSON

Si possono togliere dall'analisi le entità che non interessano (per esempio tutte le tabelle `piatto`), anche un gruppo intero alla volta.
Alla fine salva tutto in un file Excel e un file `.sql` con gli `ALTER TABLE` proposti, da controllare a mano.

Il codice dell'analisi è nel modulo Python `sqlfk` (cartella `sqlfk/` accanto al notebook): lo stesso usato dall'**app desktop** (vedi il README). Il notebook lo richiama sezione per sezione e spiega cosa fa.

**Come usarlo:**

- **con l'interfaccia grafica:** esegui tutte le celle (menu *Run → Run All Cells*); l'interfaccia compare nell'ultima sezione;
- **senza interfaccia:** modifica la cella *Configurazione* ed esegui le celle in ordine: ogni sezione mostra i suoi risultati.

Pacchetti necessari: si installano con la cella *0. Installazione* qui sotto (`sqlglot` è facoltativo ma consigliato).

## 0. Installazione (solo la prima volta)

Installa le librerie necessarie. Va eseguita una sola volta: poi si può saltare. Dopo l'installazione **riavvia il kernel** (in VS Code: *Restart*; in Jupyter: *Kernel → Restart Kernel*) e ricarica la pagina, altrimenti l'interfaccia grafica può mostrare *"Error displaying widget"*.

Se compare un errore come *"Nessuna connessione selezionata"*, il notebook è aperto con il kernel sbagliato: in alto a destra scegli il kernel **Python 3** (non SQL).

In [ ]:
%pip install pandas openpyxl sqlglot pyvis ipython scipy networkx
%pip install -U "ipywidgets>=8"

## 1. Configurazione

In [ ]:
from pathlib import Path

# Cartella con i file .sql (di default i file di prova in "esempi").
# Per il tuo database, ad esempio: Path(r"C:\Users\TUO_UTENTE\Desktop\sql_files")
cartella = Path("esempi")

# Dialetto SQL: "mysql", "tsql" (SQL Server), "oracle", "postgres", ...
DIALETTO = "mysql"

# Colonne troppo generiche per essere collegamenti: compaiono ovunque.
# Aggiungi qui quelle che nel risultato ti sembrano "rumore".
GENERICHE = {
    "id", "nome", "descrizione", "descr", "note", "data", "stato", "tipo",
    "codice", "valore", "importo", "created_at", "updated_at", "deleted_at",
    "data_ins", "data_mod", "utente_ins", "utente_mod", "attivo", "flag",
}

# Soglia di somiglianza tra nomi (da 0 a 1): più alta = meno risultati ma più precisi
SOGLIA_SIMILI = 0.85

# Entità da togliere dall'analisi (si possono togliere anche dall'interfaccia grafica).
# Scrivendo "piatto" vengono tolte tutte le tabelle di quell'entità: piatti, piatto_ingredienti, tbl_piatti...
ENTITA_DA_RIMUOVERE = []

# Gruppi di entità: quanto devono somigliarsi i nomi di due tabelle (da 0 a 1) per stare nello stesso gruppo.
# Più alta = gruppi più piccoli e più omogenei.
SOGLIA_GRUPPI = 0.75

# Sinonimi: gruppi di parole che indicano la stessa entità (plurali e accenti sono già riconosciuti)
# Esempio: SINONIMI = ["comune = citta = paese", "cliente = customer"]
SINONIMI = []

# Regole di integrità referenziale scritte negli ALTER/CREATE TABLE proposti:
# "RESTRICT", "CASCADE", "SET NULL", "SET DEFAULT", "NO ACTION" oppure "" (nessuna clausola)
ON_DELETE = "RESTRICT"
ON_UPDATE = "CASCADE"

# Parole che nei nomi delle tabelle non indicano un'entità (prefissi e suffissi tecnici)
PAROLE_NON_ENTITA = {"tbl", "tab", "tb", "t", "tmp", "temp", "bak", "backup", "old", "new", "x",
                     "ext", "cfg", "web", "log"}

print("Cartella esiste:", cartella.exists())
print("File .sql trovati:", len(list(cartella.glob("*.sql"))))

## 2. Il modulo sqlfk

Tutte le funzioni dell'analisi sono in due file:

- `sqlfk/analisi.py`: lettura dei file, gruppi di entità, chiavi primarie, collegamenti, salvataggio;
- `sqlfk/grafici.py`: gli schemi grafici interattivi.

Qui vengono importati come `A` e `G`. Le impostazioni della *Configurazione* vengono raccolte in `imp`, che viene passato alle funzioni che ne hanno bisogno.

Funzioni di supporto usate ovunque (in `sqlfk/analisi.py`): `togli_commenti`, `pulisci`, `nome_tabella`, `chiusura_parentesi`, `split_virgole`, `e_vincolo`, `colonne_regex`, `pk_da_body` per leggere i `CREATE TABLE`; `normalizza` (`COD_ART` → `codart`), `tipo_base` (`INT(11)` → `INT`), `singolari` (`province` → `provincia`, `banchi` → `banco`, `categories` → `category`), `chiavi_entita` (singolari, accenti e sinonimi), `riferimento` (`id_articolo` → `articolo`) e `parti_nome` (`OrdiniRighe` → `ordini`, `righe`) per confrontare i nomi.

In [ ]:
import importlib
import sys
import webbrowser

import pandas as pd
from IPython.display import display

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))      # il modulo sqlfk è accanto al notebook
import sqlfk.analisi as A
import sqlfk.grafici as G
import sqlfk.ontologia as O
import sqlfk.modello as MR
import sqlfk.regole as REG
importlib.reload(A)                          # se il modulo è stato modificato, ricaricalo
importlib.reload(G)
importlib.reload(O)
importlib.reload(MR)
importlib.reload(REG)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

imp = A.Impostazioni(dialetto=DIALETTO, generiche=GENERICHE, soglia_simili=SOGLIA_SIMILI,
                     soglia_gruppi=SOGLIA_GRUPPI, parole_non_entita=PAROLE_NON_ENTITA,
                     on_delete=ON_DELETE, on_update=ON_UPDATE, sinonimi=SINONIMI)
if not A.HA_SQLGLOT:
    print("sqlglot non installato: uso solo il parser interno (pip install sqlglot)")

## 3. Lettura dei file ed estrazione di tabelle e colonne

`leggi_cartella` legge tutti i file `.sql` e restituisce le colonne trovate, le chiavi primarie dichiarate, le **chiavi esterne già dichiarate** (dentro i `CREATE TABLE` o con `ALTER TABLE ... ADD FOREIGN KEY`) e i `CREATE TABLE` che non è riuscita a leggere. Il file `fk_candidate.sql` creato dal notebook viene ignorato, perché contiene solo proposte.

Codice: `A.leggi_cartella` (e `A.leggi_testi`).

In [ ]:
df_schema, pk_dichiarate, fk_dichiarate, errori = A.leggi_cartella(cartella, imp.dialetto)

print("Tabelle trovate:", df_schema["tabella"].nunique())
print("Colonne trovate:", len(df_schema))
print("Chiavi esterne già dichiarate:", len(fk_dichiarate))
print(df_schema["metodo"].value_counts().to_string())
if errori:
    print("\nErrori di lettura dei file:")
    display(pd.DataFrame(errori))
display(df_schema)

## 4. Gruppi di entità (clustering dei nomi)

Con centinaia di tabelle è utile vedere quali appartengono alla stessa entità: `ordini`, `ordini_righe`, `tbl_ordini` e `ordine_storico` parlano tutte di ordini. Questa sezione divide le tabelle in **gruppi** in base alla somiglianza dei nomi, come un clustering.

Come funziona:

1. il nome di ogni tabella viene diviso in parole (`tbl_ordini_righe` → `tbl`, `ordini`, `righe`); le parole tecniche in `PAROLE_NON_ENTITA` (`tbl`, `tmp`, `bak`...) vengono scartate e le altre ridotte alla radice (`ordini` → `ordin`);
2. la **somiglianza** tra due tabelle è il valore più alto tra:
   - se le prime parole sono **la stessa parola**, anche al plurale (italiano e inglese), con o senza accenti, o sinonimi (`SINONIMI`): 1,0;
   - altrimenti quanto è lungo l'**inizio comune** delle loro prime parole, rispetto alla più lunga (`ordini_righe` e `ordine_storico` → `ordin` e `ordin`: 1,0; `anagrafica` e `anagrafiche` → `anagrafic` e `anagrafich`, 8 lettere comuni su 10: 0,8). Conta solo l'inizio: `agenti` e `pagamenti` non si somigliano anche se `agent` sta dentro `pagament`;
   - la **quota di parole in comune** (`piatti` e `piatto_ingredienti` hanno in comune 1 parola su 2: 0,5);
3. le tabelle vengono unite in gruppi con un **clustering gerarchico** (legame medio): due gruppi si uniscono solo se, in media, le loro tabelle si somigliano almeno quanto `SOGLIA_GRUPPI`. Il legame medio evita le "catene": `piatto_ingredienti` non basta a mettere `piatti` e `ingredienti` nello stesso gruppo.

Ogni gruppo prende il nome della prima parola più frequente tra le sue tabelle. Le tabelle che non somigliano a nessun'altra restano in un gruppo da sole.

Codice: `A.raggruppa_tabelle`, `A.parole_entita`, `A.inizio_comune`.

In [ ]:
df_gruppi, gruppo_di = A.raggruppa_tabelle(df_schema["tabella"].unique(), imp)
print("Gruppi con almeno 2 tabelle:", (df_gruppi["n_tabelle"] > 1).sum(),
      "| tabelle da sole:", (df_gruppi["n_tabelle"] == 1).sum())
display(df_gruppi)

## 5. Schema iniziale: com'è collegato il database

Prima di cercare le chiavi esterne mancanti, questa sezione mostra il database **così com'è nei file**:

- un riepilogo: tabelle, colonne, chiavi primarie (su una colonna, composte, assenti), chiavi esterne già dichiarate, tabelle non collegate, gruppi;
- l'elenco delle **chiavi esterne già dichiarate** (spesso nessuna);
- lo schema grafico `schema_iniziale.html`, che si apre nel browser: ogni casella è una tabella, **colorata in base al suo gruppo** (sezione 4); le linee grigie tratteggiate uniscono le tabelle dello stesso gruppo, le frecce blu spesse sono le chiavi esterne dichiarate. Passando il mouse su una casella si vedono gruppo e colonne (`[PK]` = chiave primaria).

Codice: `A.riepilogo_iniziale` e `G.schema_iniziale`.

In [ ]:
display(A.riepilogo_iniziale(df_schema, pk_dichiarate, fk_dichiarate, df_gruppi))
if fk_dichiarate.empty:
    print("Nessuna chiave esterna dichiarata: nei file le tabelle non sono collegate tra loro.")
else:
    display(fk_dichiarate)

file_iniziale = cartella / "schema_iniziale.html"
G.schema_iniziale(df_schema, pk_dichiarate, fk_dichiarate, gruppo_di, file_iniziale)
webbrowser.open(file_iniziale.resolve().as_uri())
print("Salvato:", file_iniziale)

## 6. Togliere entità dall'analisi

Scrivendo il nome di un'entità (per esempio `piatto`) vengono tolte tutte le sue tabelle: quelle in cui una parte del nome ha la stessa radice (`piatti`, `piatto_ingredienti`, `tbl_piatti`, `menu_piatti`).

Plurali italiani e inglesi (`provincia`/`province`, `banco`/`banchi`, `category`/`categories`) e accenti (`città`/`citta`) sono riconosciuti da soli; i sinonimi (`comune = citta`) si indicano in `SINONIMI`.

Le parole si scrivono in `ENTITA_DA_RIMUOVERE` nella *Configurazione*, oppure nella scheda *Togli entità* dell'interfaccia grafica (dove si può togliere anche un intero gruppo della sezione 4), che mostra prima l'elenco delle tabelle trovate e permette di tenerne alcune. Le tabelle tolte non vengono analizzate e non compaiono nei risultati né nello schema; i file `.sql` non vengono modificati.

Codice: `A.tabelle_entita` e `A.filtra_schema`.

In [ ]:
rimosse = set()
for parola in ENTITA_DA_RIMUOVERE:
    trovate = A.tabelle_entita(df_schema["tabella"].unique(), parola, imp.sinonimi)
    print(f"{parola}: {len(trovate)} tabelle tolte -> {', '.join(trovate) or 'nessuna'}")
    rimosse |= set(trovate)

df_analisi, pk_analisi = A.filtra_schema(df_schema, pk_dichiarate, rimosse)
print("Tabelle da analizzare:", df_analisi["tabella"].nunique(), "| tolte:", len(rimosse))

## 7. Chiavi primarie

`prepara` raccoglie le informazioni usate da tutte le analisi successive. Usa le chiavi primarie dichiarate nei file; se una tabella non ne ha, prova a indovinarla (`id`, `id_tabella`, `tabella_id`). La colonna **n colonne PK** indica se la chiave primaria è composta (più di una colonna).

Codice: `A.prepara`.

In [ ]:
S = A.prepara(df_analisi, pk_analisi, fk_dichiarate, imp)
print("Tabelle senza chiave primaria:", (S.riepilogo["chiave primaria"] == "-").sum())
print("Tabelle con chiave primaria composta:", len(S.pk_composta))
display(S.riepilogo)

## 8. Colonne con lo stesso nome in più tabelle

L'indizio più forte. La colonna **"è PK in"** dice in quale tabella quella colonna è chiave primaria: di solito è la tabella "principale" a cui puntano le altre. **"parte di PK composta in"** dice in quali tabelle la colonna fa parte di una chiave primaria su più colonne.

Codice: `A.trova_stesso_nome`.

In [ ]:
stesso_nome = A.trova_stesso_nome(S)
print("Colonne con lo stesso nome in più tabelle:", len(stesso_nome))
display(stesso_nome)

## 9. Colonne con nomi simili (non identici)

Ci saranno falsi positivi: tieni soprattutto le coppie con **stesso tipo = True**. Se i risultati sono troppi, alza `SOGLIA_SIMILI`. Con centinaia di tabelle questa è la parte più lenta (fino a un minuto circa).

Codice: `A.trova_nomi_simili`.

In [ ]:
df_simili = A.trova_nomi_simili(S, imp)
print("Coppie di colonne con nomi simili:", len(df_simili))
display(df_simili)

## 10. Chiavi esterne candidate

Per ogni colonna cerca la tabella a cui potrebbe puntare:

- **alta**: la colonna ha lo stesso nome della chiave primaria di un'altra tabella
- **media**: il nome rimanda al nome di una tabella (`id_articolo` → `articoli`)
- **alta (PK composta)**: la tabella contiene **tutte** le colonne della chiave primaria composta di un'altra tabella (`id_ordine` + `num_riga` → `ordini_righe`). Il vincolo proposto è su più colonne insieme.

Se due tabelle hanno la stessa chiave primaria composta, il collegamento viene proposto in entrambe le direzioni: tieni quello che va verso la tabella principale.

**già dichiarata = True** indica una chiave esterna che esiste già nei file.

**tipi ok = False** significa che il vincolo fallirebbe: prima va sistemato il tipo.

Codice: `A.trova_fk`.

In [ ]:
df_fk, sospette = A.trova_fk(S)
print("FK candidate:", len(df_fk), "| di cui affidabilità alta:", (df_fk["affidabilità"] == "alta").sum())
display(df_fk)
print("Colonne che sembrano chiavi ma senza tabella corrispondente:", len(sospette))
display(sospette)

## 11. Tabelle con chiave primaria composta: collegamenti colonna per colonna

Per ogni tabella con una chiave primaria su più colonne, ogni colonna della chiave viene analizzata da sola. Per esempio, se la tabella `ordini` ha come chiave primaria `codice, id, nome`, per ognuna delle tre colonne:

1. cerca una **tabella con il nome del campo** (`codice` → tabella `codice` o `codici`; `id_cliente` → tabella `clienti`) e verifica se il campo è presente in quella tabella e se lì è chiave primaria;
2. se non la trova, cerca le tabelle in cui quel campo è **chiave primaria** (da sola o come parte di una chiave composta);
3. se non trova nemmeno quelle, elenca le tabelle che hanno **lo stesso campo** come colonna normale.

Per le colonne generiche (`id`, `nome`, `codice`... vedi `GENERICHE`) vale solo il passo 1: altrimenti verrebbero collegate a centinaia di tabelle.

| Affidabilità | Quando |
|---|---|
| **alta** | il campo è chiave primaria della tabella trovata |
| **media** | il campo è presente ma non è chiave primaria, oppure la tabella col nome del campo ha una chiave primaria diversa, oppure il campo fa parte di una chiave composta altrove |
| **bassa** | solo lo stesso nome di colonna in un'altra tabella, o tabella trovata ma senza corrispondenza |

Codice: `A.collegamenti_pk_composte` e `A.descrivi_pk_composte`.

In [ ]:
df_pk_composte = A.collegamenti_pk_composte(S)
print("Tabelle con chiave primaria composta:", len(S.pk_composta), "\n")
print(A.descrivi_pk_composte(df_pk_composte))
display(df_pk_composte)

## 12. Qualità dello schema: regole di buona progettazione

Controlli tratti dalla teoria della progettazione dei database relazionali, ricavati dalla sola struttura (`CREATE TABLE`). Senza i dati non si possono dimostrare: sono **avvisi da verificare**. Si basano sulle chiavi esterne già dichiarate e, per ogni colonna, sulla candidata migliore ad affidabilità alta con tipi compatibili.

- **Integrità dell'entità**: ogni tabella deve avere una chiave primaria (valori presenti e diversi tra loro). Per le tabelle che non ce l'hanno viene proposta una chiave artificiale, anche nel `CREATE TABLE` (commentata).
- **Integrità referenziale**: ogni chiave esterna proposta ha le regole `ON DELETE` e `ON UPDATE` scelte in *Configurazione* (`ON_DELETE`, `ON_UPDATE`). `SET NULL` non si può usare se la colonna è `NOT NULL`, e MySQL non accetta `SET DEFAULT`: in questi casi si usa `RESTRICT` e lo si segnala.
- **Tipo di relazione** (colonne *relazione* e *obbligatoria* delle FK candidate): **1:1** se la chiave esterna è anche `UNIQUE` o coincide con la chiave primaria, altrimenti **1:N**; **obbligatoria** se le sue colonne sono `NOT NULL`. Le **tabelle ponte**, la cui chiave primaria è fatta di chiavi esterne verso tabelle diverse, realizzano una relazione **N:M**.
- **Normalizzazione**:
  - **1NF**: campi composti (`generalita`, `nome_cognome`, `indirizzo_completo`), tipi con più valori (`SET`, `JSON`) e gruppi ripetuti (`telefono1`, `telefono2`);
  - **2NF**: in una tabella con chiave composta, una colonna che si trova già nella tabella puntata da una **parte** della chiave (per esempio `forniture.citta`, che dipende solo dal fornitore);
  - **3NF**: una colonna copiata dalla tabella puntata da una chiave esterna non chiave (per esempio `fatture.ragione_sociale`, che dipende dal cliente): è una dipendenza transitiva.
- **Relazioni ridondanti**: un collegamento diretto che ripete un percorso già esistente (per esempio `comuni → regioni` quando c'è già `comuni → province → regioni`). Nel `CREATE TABLE` viene commentato.

Codice: `A.controlli_progettazione`, `A.tipo_relazione`, `A.clausola_regole`.

In [ ]:
Q = A.controlli_progettazione(S, df_fk, fk_dichiarate)
display(Q.riepilogo)
print(f"Regole delle chiavi esterne: ON DELETE {imp.on_delete or '-'}, ON UPDATE {imp.on_update or '-'}")
for titolo, tabella in [("Tabelle senza chiave primaria", Q.senza_pk), ("Relazioni N:M (tabelle ponte)", Q.ponti),
                        ("Normalizzazione", Q.normalizzazione), ("Relazioni ridondanti", Q.ridondanti)]:
    print(f"\n{titolo}: {len(tabella)}")
    if len(tabella):
        display(tabella)

## 13. Tabelle isolate

Tabelle che non compaiono in nessun collegamento trovato: sono quelle su cui indagare a mano.

Codice: `A.trova_isolate`.

In [ ]:
isolate = A.trova_isolate(S, stesso_nome, df_simili, df_fk, df_pk_composte)
print("Tabelle senza nessun collegamento trovato:", len(isolate))
for t in isolate:
    print(f"  {t}: {', '.join(S.col_per_tab[t])}")

## 14. Analisi completa

`analizza` esegue in un colpo solo tutte le analisi delle sezioni 7-13. La usano l'interfaccia grafica e l'app ogni volta che si caricano i file o si tolgono entità. Le coppie di nomi simili, la parte lenta, si possono calcolare una volta sola e riusare: togliendo tabelle basta scartare le coppie che le contengono.

Codice: `A.analizza`.

In [ ]:
originali = A.create_originali_cartella(cartella, imp.dialetto)   # i CREATE TABLE dei file
R = A.analizza(df_schema, pk_dichiarate, rimosse, simili_tutti=df_simili,
              fk_dichiarate=fk_dichiarate, gruppi=df_gruppi, imp=imp, errori=errori, originali=originali)
print(f"{len(R.S.col_per_tab)} tabelle analizzate, {len(R.fk)} FK candidate, "
      f"{len(R.S.pk_composta)} tabelle con chiave primaria composta, {len(R.isolate)} isolate")

## 15. Diagnostica: errori e avvisi come un compilatore

Tutti i controlli precedenti in un unico elenco, come l'output di un compilatore: per ogni problema **file e riga**, **gravità**, **codice**, messaggio e suggerimento.

```
esempio_5.sql:44: errore E010: la tabella log_accessi non ha una chiave primaria
    suggerimento: aggiungere una chiave artificiale: id_log_accessi INT NOT NULL AUTO_INCREMENT PRIMARY KEY
```

| Gravità | Significato | Codici |
|---|---|---|
| **errore** | qualcosa di sbagliato da correggere | E001-E003 lettura dei file (parentesi non chiusa, tabella o colonna definita due volte); E010-E011 chiave primaria mancante o su colonna inesistente; E020-E025 chiavi esterne dichiarate sbagliate (tabella o colonna inesistente, non puntano alla chiave, tipi diversi, tabella partizionata) |
| **avviso** | probabile difetto di progettazione, da verificare | W026 chiave esterna su tabella MyISAM; W030 chiave esterna mancante; W031 tipi incompatibili; W033 colonna da chiave senza tabella; W034 relazione ridondante; W101-W103 prima forma normale; W201 seconda; W301 terza |
| **info** | osservazione | I032 possibile chiave esterna ad affidabilità media; I035 tabella isolata; I036 tabella ponte N:M; I038 relazione facoltativa |

Ogni codice appartiene a una **regola di progettazione** (R0-R8: codice corretto, integrità dell'entità, integrità referenziale, collegamenti dichiarati, niente relazioni ridondanti, cardinalità, prima, seconda e terza forma normale). Con `con_regole=True` il report mostra la regola violata; `REG.spiega("W301")` la spiega con un esempio sbagliato e uno corretto.

Lo stesso controllo si può fare dal terminale, come un compilatore vero (codice di uscita 1 se ci sono errori):

```bash
python -m sqlfk.verifica cartella_dei_file --senza-info --regole
python -m sqlfk.verifica --spiega W301
```

Codice: `A.diagnostica`, `A.testo_diagnostica`; da terminale `sqlfk/verifica.py`.

In [ ]:
print(A.testo_diagnostica(R.diagnostica, con_info=False, con_regole=True))
display(R.diagnostica)
print(REG.spiega("E010"))

## 16. Aree, sottogruppi, collegamenti tra sottogruppi e categorie

I gruppi della sezione 4 sono fatti solo con i nomi. Qui si sale di un livello:

- **sottogruppi**: i gruppi per nome della sezione 4;
- **aree**: sottogruppi molto collegati tra loro dalle chiavi esterne (dichiarate e trovate ad affidabilità alta). Si trovano con un rilevamento di comunità (modularità): i sottogruppi con molti collegamenti tra loro e pochi con il resto formano un'area. L'area prende il nome del sottogruppo con più collegamenti;
- **tabelle centrali**: tabelle puntate da moltissime altre di sottogruppi diversi (per esempio una tabella `collection` presente in tutte le chiavi). Se contassero come gli altri collegamenti unirebbero tutto in un'unica area, quindi nella divisione pesano poco;
- **collegamenti tra sottogruppi** (con il numero di chiavi esterne, il tipo 1:N, 1:1 o N:M e le colonne) e **collegamenti tra aree**;
- **categoria** di ogni tabella, ricavata dalla struttura: codifica (lookup), anagrafica, movimento, dettaglio, ponte, storico, dipendente, isolata. È un'indicazione da verificare, non una certezza.

Le tabelle senza nessun collegamento finiscono nell'area *senza collegamenti*.

Codice: `A.aree_e_categorie` (chiamata da `analizza`, risultato in `R.aree`).

In [ ]:
X = R.aree
print(f"{len(X.aree)} aree, {len(X.sottogruppi)} sottogruppi; tabelle centrali: {', '.join(X.centrali) or 'nessuna'}")
display(X.aree)
display(X.sottogruppi)
print("Collegamenti tra aree")
display(X.collegamenti_aree)
print("Collegamenti tra sottogruppi (prima quelli tra aree diverse)")
display(X.collegamenti)
print("Categorie delle tabelle:", X.tabelle["categoria"].value_counts().to_dict())
display(X.tabelle)

# albero area -> sottogruppo -> tabelle (categoria)
for area in X.aree["area"]:
    print(f"\nAREA {area}")
    for r in X.sottogruppi[X.sottogruppi["area"] == area].to_dict("records"):
        voci = [f"{t} ({X.categoria_di[t]}{', centrale' if t in X.centrali else ''})" for t in r["tabelle"].split(", ")
                if t in X.categoria_di]
        print(f"  {r['sottogruppo']}: " + ", ".join(voci))

## 17. Query sui dati

Dai file `.sql` si costruisce un database **SQLite in memoria**: le tabelle dei `CREATE TABLE` (con le colonne aggiunte da `ALTER TABLE ... ADD COLUMN`) e i dati degli `INSERT INTO` e dei blocchi `COPY ... FROM stdin` dei dump di PostgreSQL.

Le query si scrivono in SQLite oppure nel dialetto impostato (`DIALETTO`): se SQLite non le capisce vengono tradotte con sqlglot (per esempio `ILIKE`, `::int`, `TOP 10`). I nomi con lo schema (`public.clienti`, `vendite.libri`) funzionano.

`esegui` non esegue le istruzioni che modificano i dati (INSERT, UPDATE, DELETE, DROP...) se non con `consenti_modifiche=True`. Le modifiche cambiano solo la copia in memoria, non i file: per annullarle si ricostruisce il database.

Codice: `sqlfk/query.py` (`QY.Database`, `db.esegui`, `QY.testo_risultato`). Da terminale: `python -m sqlfk.query cartella "SELECT ..."`.

In [ ]:
import sqlfk.query as QY
importlib.reload(QY)

file_sql = [(f.name, f.read_text(encoding="utf-8", errors="replace")) for f in sorted(cartella.glob("*.sql"))]
db = QY.Database(file_sql, df_schema, pk_dichiarate, imp.dialetto)
print(f"Database costruito in {db.secondi:.1f} s")
display(pd.DataFrame(db.conteggi()))
if db.problemi:
    print("Istruzioni di dati non caricate:")
    display(pd.DataFrame(db.problemi))

def query(testo, consenti_modifiche=False):
    """Esegue una o più query e mostra i risultati come tabelle."""
    esito = db.esegui(testo, consenti_modifiche)
    if esito.get("serve_conferma"):
        print("Queste istruzioni modificano i dati: rilancia con consenti_modifiche=True")
        for m in esito["modifiche"]:
            print("  ", m)
        return
    for r in esito["risultati"]:
        if r["tradotta"]:
            print("Eseguita come:", r["tradotta"])
        if r["errore"]:
            print("Errore:", r["errore"])
        elif r["colonne"]:
            print(f"{r['totale']} righe")
            display(pd.DataFrame(r["righe"], columns=r["colonne"]))
        else:
            print("Righe modificate:", r["modificate"])

# scrivi qui la tua query
query("SELECT name AS tabella FROM sqlite_master WHERE type = 'table' ORDER BY name LIMIT 5")

## 18. Ontologia OWL e mapping R2RML

Dallo schema analizzato si genera un'ontologia **OWL** in formato Turtle (`ontologia.ttl`, apribile in Protégé) e un mapping **R2RML** (`mapping_r2rml.ttl`, lo standard W3C per trasformare i dati del database in RDF con strumenti come Ontop o RMLMapper).

| Nel database | Nell'ontologia |
|---|---|
| tabella `province` | classe `:Provincia` (al singolare: regole sicure dei plurali e indizi dello schema come `id_provincia`) |
| colonna | proprietà di dato con il tipo XSD |
| chiave esterna | proprietà d'oggetto `:haProvincia` (`has...` se i nomi sono in inglese) |
| chiave primaria | `owl:hasKey` |
| `NOT NULL` | cardinalità minima 1 |
| relazione 1:1 | proprietà funzionale e funzionale inversa |
| tabella ponte senza attributi propri | proprietà d'oggetto N:M diretta, con la sua inversa |

Si usano le chiavi esterne dichiarate e quelle trovate ad affidabilità alta (annotate come *da verificare*); relazioni ridondanti e tabelle tolte sono escluse. `BASE` è l'IRI di base dell'ontologia.

Codice: `sqlfk/ontologia.py` (`O.genera_owl`, `O.genera_r2rml`); da terminale `python -m sqlfk.ontologia cartella --base http://...`.

In [ ]:
BASE = O.base_predefinita(cartella.resolve().name)    # oppure per esempio "http://example.org/negozio/"
owl, modello_onto = O.genera_owl(R, BASE, cartella.resolve().name)
(cartella / "ontologia.ttl").write_text(owl, encoding="utf-8")
(cartella / "mapping_r2rml.ttl").write_text(O.genera_r2rml(R, BASE, modello_onto), encoding="utf-8")
riassunto = O.riepilogo(modello_onto)
print(f"{riassunto['classi']} classi, {riassunto['proprieta_dato']} proprietà di dato, "
      f"{riassunto['proprieta_oggetto']} proprietà d'oggetto ({riassunto['proposte']} proposte), "
      f"{riassunto['relazioni_nm']} relazioni N:M, nomi in {riassunto['lingua']}")
print("Salvati:", cartella / "ontologia.ttl", "e", cartella / "mapping_r2rml.ttl")
display(pd.DataFrame(riassunto["tabelle"]))

## 19. Modello relazionale (schema logico) ed esportazioni

Lo schema logico del database, una riga per tabella: `Fatture (numero (pk), data, id_cliente (fk → clienti), importo)`. Usa le chiavi esterne dichiarate e quelle trovate ad affidabilità alta (`tipo="iniziale"` per le sole dichiarate), senza le relazioni ridondanti.

Si può esportare in: testo (`.txt`), Markdown (`.md`), **Mermaid** (`.mmd`, lo disegnano GitHub, Notion e mermaid.live), **PlantUML** (`.puml`), **Graphviz** (`.dot`: `dot -Tpng modello_relazionale.dot -o modello.png`), **DBML** (`.dbml`, per dbdiagram.io) e JSON. L'immagine PNG o SVG del diagramma si scarica dall'app (pagine *Modello relazionale* e *Schemi grafici*).

Codice: `sqlfk/modello.py` (`MR.struttura`, `MR.esporta`); da terminale `python -m sqlfk.modello cartella --formato mermaid`.

In [ ]:
modello_rel = MR.struttura(R, gruppo_di, tipo="collegamenti")
print(MR.esporta(modello_rel, "testo"))
for formato, estensione in MR.FORMATI.items():
    (cartella / f"modello_relazionale.{estensione}").write_text(MR.esporta(modello_rel, formato), encoding="utf-8")
print("Salvati i file modello_relazionale.* in", cartella)

## 20. Salvataggio

Crea nella cartella dei file `.sql`:

- `possibili_collegamenti.xlsx` con un foglio per ogni analisi (compresi gruppi e chiavi esterne già dichiarate)
- `fk_candidate.sql` con gli `ALTER TABLE` proposti (**da controllare prima di eseguirli**)
- `diagnostica.txt` con il report di errori e avvisi e le regole violate (sezione 15)
- `create_table_con_fk.sql` con i `CREATE TABLE` dei tuoi file e le chiavi esterne già scritte dentro: sono **attive** quelle ad affidabilità alta con tipi compatibili (una sola per colonna) e quelle già dichiarate con `ALTER TABLE`; le altre sono **commentate** con `--` e il motivo (affidabilità media, tipi diversi, alternative sulla stessa colonna, tabelle MyISAM o partizionate che in MySQL non accettano chiavi esterne). Le tabelle sono in ordine: prima quelle a cui puntano le altre.

Codice: `A.salva`, `A.testo_alter`, `A.testo_create`.

In [ ]:
for file in A.salva(R, cartella, originali, imp.dialetto):
    print("Salvato:", file)
(cartella / "diagnostica.txt").write_text(A.testo_diagnostica(R.diagnostica, con_regole=True), encoding="utf-8")
print("Salvato:", cartella / "diagnostica.txt")

## 21. Schema grafico dei collegamenti trovati

Crea `schema_collegamenti.html` e lo apre nel browser. Ogni casella è una tabella (passa il mouse per vedere le colonne, `[PK]` = chiave primaria); le frecce verdi sono collegamenti ad affidabilità alta, quelle arancioni tratteggiate media. Le tabelle grigie sono isolate. Puoi trascinare le caselle e fare zoom.

Codice: `G.schema_collegamenti`.

In [ ]:
file_collegamenti = cartella / "schema_collegamenti.html"
G.schema_collegamenti(R, file_collegamenti)
webbrowser.open(file_collegamenti.resolve().as_uri())
print("Salvato:", file_collegamenti)

## 22. Interfaccia grafica

Esegui questa cella (dopo tutte le precedenti) per usare il notebook senza modificare il codice:

- in alto scegli la **cartella** dei file `.sql` e il **dialetto**, poi premi **Carica e analizza**;
- **Panoramica**: il database com'è prima dell'analisi (riepilogo e chiavi esterne già dichiarate) e il pulsante per aprire lo **schema iniziale**;
- **Gruppi**: i gruppi di entità con nomi simili; scegli un gruppo per vedere le sue tabelle e, se vuoi, toglierlo dall'analisi con un clic;
- **Tabelle**: tutte le tabelle con la loro chiave primaria, con un campo per filtrare;
- **PK composte**: scegli una tabella con chiave primaria composta e vedi, colonna per colonna, a cosa si collega;
- **FK candidate**: le chiavi esterne proposte, filtrabili per affidabilità, tipi e nome;
- **Altri risultati**: colonne con lo stesso nome, nomi simili, colonne da verificare, tabelle isolate;
- **Togli entità**: scrivi un'entità (per esempio `piatto`), premi **Cerca tabelle**, deseleziona quelle da tenere e premi **Togli selezionate**: l'analisi si aggiorna da sola. Le tabelle tolte si possono rimettere;
- **Salva**: crea il file Excel, il file `.sql` e lo schema grafico dei collegamenti trovati.

Le tabelle a video mostrano al massimo 200 righe: i risultati completi sono nel file Excel.

In [ ]:
import ipywidgets as w
from IPython.display import HTML

stato = {"df_schema": None, "pk": {}, "fk": None, "gruppi": None, "gruppo_di": {},
         "simili": None, "rimosse": set(rimosse), "R": None, "cartella": Path(cartella)}
largo = {"description_width": "initial"}

# --- controlli
txt_cartella = w.Text(value=str(cartella), description="Cartella dei file .sql:", style=largo,
                      layout=w.Layout(width="55%"))
dd_dialetto = w.Dropdown(options=["mysql", "tsql", "oracle", "postgres", "sqlite"],
                         value=imp.dialetto, description="Dialetto:", style=largo)
btn_analizza = w.Button(description="Carica e analizza", button_style="primary")
lbl_stato = w.HTML()

out_panoramica = w.Output()
btn_iniziale = w.Button(description="Apri schema iniziale")
out_iniziale = w.Output()

dd_gruppo = w.Dropdown(description="Gruppo:", options=[], style=largo, layout=w.Layout(width="50%"))
btn_togli_gruppo = w.Button(description="Togli questo gruppo", button_style="danger")
lbl_gruppo = w.HTML()
out_gruppi = w.Output()

txt_cerca = w.Text(placeholder="parte del nome", description="Filtra tabelle:", style=largo)
out_tabelle = w.Output()

dd_pk = w.Dropdown(description="Tabella:", options=[], style=largo, layout=w.Layout(width="50%"))
out_pk = w.Output()

dd_aff = w.Dropdown(options=["tutte", "alta", "media"], description="Affidabilità:", style=largo)
chk_tipi = w.Checkbox(description="solo tipi compatibili", value=False, indent=False)
txt_fk = w.Text(placeholder="parte del nome", description="Tabella:", style=largo)
out_fk = w.Output()

out_altri = w.Output()

txt_entita = w.Text(placeholder="es. piatto", description="Entità da togliere:", style=largo,
                    continuous_update=False)   # cerca quando si preme Invio
btn_cerca_entita = w.Button(description="Cerca tabelle")
lbl_entita = w.HTML()
sel_trovate = w.SelectMultiple(options=[], rows=10, description="Trovate:", style=largo,
                               layout=w.Layout(width="60%"))
btn_togli = w.Button(description="Togli selezionate", button_style="danger")
sel_tolte = w.SelectMultiple(options=[], rows=10, description="Tolte:", style=largo,
                             layout=w.Layout(width="60%"))
btn_rimetti = w.Button(description="Rimetti selezionate")
btn_rimetti_tutte = w.Button(description="Rimetti tutte")

btn_salva = w.Button(description="Salva Excel e SQL")
btn_schema = w.Button(description="Crea e apri schema grafico")
out_salva = w.Output()


# --- aggiornamento delle viste
def mostra(out, df, titolo=None, vuoto="Nessun risultato."):
    with out:
        if titolo:
            display(HTML(f"<h4>{titolo}</h4>"))
        if df is None or len(df) == 0:
            print(vuoto)
        else:
            if len(df) > 200:
                print(f"{len(df)} righe: mostrate le prime 200, tutte nel file Excel.")
            display(df.head(200))


def contiene(serie, testo):
    return serie.astype(str).str.lower().str.contains(testo.strip().lower(), regex=False)


def vista_panoramica():
    out_panoramica.clear_output()
    with out_panoramica:
        display(A.riepilogo_iniziale(stato["df_schema"], stato["pk"], stato["fk"], stato["gruppi"]))
        if stato["fk"].empty:
            print("Nessuna chiave esterna dichiarata: nei file le tabelle non sono collegate tra loro.")
    if not stato["fk"].empty:
        mostra(out_panoramica, stato["fk"], "Chiavi esterne già dichiarate")


def vista_gruppi(_=None):
    out_gruppi.clear_output()
    if stato["gruppi"] is None:
        return
    df = stato["gruppi"]
    if dd_gruppo.value in (None, "(tutti)"):
        mostra(out_gruppi, df)
        return
    tabelle = sorted(t for t, g in stato["gruppo_di"].items() if g == dd_gruppo.value)
    mostra(out_gruppi, pd.DataFrame({"tabella": tabelle,
                                     "tolta": [t in stato["rimosse"] for t in tabelle]}))


def vista_tabelle(_=None):
    out_tabelle.clear_output()
    if stato["R"] is None:
        return
    df = stato["R"].S.riepilogo
    if txt_cerca.value.strip():
        df = df[contiene(df["tabella"], txt_cerca.value)]
    mostra(out_tabelle, df)


def vista_pk(_=None):
    out_pk.clear_output()
    if stato["R"] is None:
        return
    df = stato["R"].pk_composte
    if df.empty:
        with out_pk:
            print("Nessuna tabella con chiave primaria composta.")
        return
    scelta = None if dd_pk.value in (None, "(tutte)") else dd_pk.value
    with out_pk:
        print(A.descrivi_pk_composte(df, scelta))
    mostra(out_pk, df if scelta is None else df[df["tabella"] == scelta])


def vista_fk(_=None):
    out_fk.clear_output()
    if stato["R"] is None:
        return
    df = stato["R"].fk
    if dd_aff.value != "tutte":
        df = df[df["affidabilità"] == dd_aff.value]
    if chk_tipi.value:
        df = df[df["tipi ok"]]
    if txt_fk.value.strip():
        df = df[contiene(df["tabella"], txt_fk.value) | contiene(df["→ tabella"], txt_fk.value)]
    with out_fk:
        print(f"{len(df)} chiavi esterne candidate")
    mostra(out_fk, df)


def vista_altri():
    out_altri.clear_output()
    R = stato["R"]
    mostra(out_altri, R.stesso_nome, "Colonne con lo stesso nome in più tabelle")
    mostra(out_altri, R.simili, "Colonne con nomi simili")
    mostra(out_altri, R.sospette, "Colonne che sembrano chiavi ma senza tabella corrispondente")
    mostra(out_altri, pd.DataFrame({"tabella": R.isolate}), "Tabelle isolate")
    mostra(out_altri, R.diagnostica, "Diagnostica: errori, avvisi e info")
    mostra(out_altri, R.qualita.riepilogo, "Qualità dello schema")
    mostra(out_altri, R.qualita.senza_pk, "Tabelle senza chiave primaria")
    mostra(out_altri, R.qualita.ponti, "Relazioni N:M (tabelle ponte)")
    mostra(out_altri, R.qualita.normalizzazione, "Normalizzazione")
    mostra(out_altri, R.qualita.ridondanti, "Relazioni ridondanti")


def aggiorna():
    R = stato["R"] = A.analizza(stato["df_schema"], stato["pk"], stato["rimosse"], stato["simili"],
                                stato["fk"], stato["gruppi"], imp)
    lbl_stato.value = (f"<b>{len(R.S.col_per_tab)}</b> tabelle analizzate, "
                       f"<b>{int((stato['gruppi']['n_tabelle'] > 1).sum())}</b> gruppi, "
                       f"<b>{len(R.S.col)}</b> colonne, "
                       f"<b>{len(R.S.pk_composta)}</b> con chiave primaria composta, "
                       f"<b>{len(R.fk)}</b> FK candidate, "
                       f"<b>{len(R.isolate)}</b> isolate, "
                       f"<b>{len(stato['rimosse'])}</b> tolte")
    scelta = dd_pk.value
    dd_pk.options = ["(tutte)"] + sorted(R.S.pk_composta)
    dd_pk.value = scelta if scelta in dd_pk.options else "(tutte)"
    sel_tolte.options = sorted(stato["rimosse"])
    vista_gruppi()
    vista_tabelle()
    vista_pk()
    vista_fk()
    vista_altri()


# --- azioni dei pulsanti
def carica(_=None):
    imp.dialetto = dd_dialetto.value
    percorso = Path(txt_cartella.value.strip())
    if not percorso.is_dir():
        lbl_stato.value = f"<span style='color:#b00020'>La cartella non esiste: {percorso}</span>"
        return
    lbl_stato.value = "Analisi in corso... con molte tabelle può richiedere un minuto."
    try:
        df, pk_d, fk_d, err = A.leggi_cartella(percorso, imp.dialetto)
        if df.empty:
            lbl_stato.value = "Nessun CREATE TABLE trovato nella cartella."
            return
        gruppi, gruppo_tab = A.raggruppa_tabelle(df["tabella"].unique(), imp)
        stato.update(df_schema=df, pk=pk_d, fk=fk_d, gruppi=gruppi, gruppo_di=gruppo_tab, cartella=percorso)
        scelta = dd_gruppo.value
        dd_gruppo.options = ["(tutti)"] + [g for g, n in zip(gruppi["gruppo"], gruppi["n_tabelle"]) if n > 1]
        dd_gruppo.value = scelta if scelta in dd_gruppo.options else "(tutti)"
        vista_panoramica()
        stato["rimosse"] &= set(df["tabella"])
        # i nomi simili si calcolano una volta sola, su tutto lo schema
        stato["simili"] = A.trova_nomi_simili(A.prepara(df, pk_d, None, imp), imp)
        aggiorna()
        if err:
            lbl_stato.value += f"<br>{len(err)} CREATE TABLE non letti (parentesi non chiusa)."
    except Exception as e:
        lbl_stato.value = f"<span style='color:#b00020'>Errore: {type(e).__name__}: {e}</span>"


def cerca_entita(_=None):
    if stato["df_schema"] is None:
        lbl_entita.value = "Prima carica i file."
        return
    parola = txt_entita.value.strip()
    tabelle = [t for t in stato["df_schema"]["tabella"].unique() if t not in stato["rimosse"]]
    trovate = A.tabelle_entita(tabelle, parola, imp.sinonimi)
    sel_trovate.options = trovate
    sel_trovate.value = tuple(trovate)
    if trovate:
        lbl_entita.value = (f"{len(trovate)} tabelle trovate per <b>{parola}</b>. "
                            "Tutte selezionate: deseleziona quelle da tenere (Cmd/Ctrl + clic), "
                            "poi premi <b>Togli selezionate</b>.")
    else:
        lbl_entita.value = f"Nessuna tabella trovata per <b>{parola}</b> (servono almeno 3 lettere)."


def togli(_=None):
    scelte = set(sel_trovate.value)
    if not scelte:
        lbl_entita.value = "Nessuna tabella selezionata."
        return
    stato["rimosse"] |= scelte
    sel_trovate.options = []
    lbl_entita.value = f"Tolte {len(scelte)} tabelle."
    aggiorna()


def togli_gruppo(_=None):
    if dd_gruppo.value in (None, "(tutti)"):
        lbl_gruppo.value = "Scegli prima un gruppo."
        return
    tabelle = {t for t, g in stato["gruppo_di"].items() if g == dd_gruppo.value} - stato["rimosse"]
    if not tabelle:
        lbl_gruppo.value = "Le tabelle di questo gruppo sono già tutte tolte."
        return
    stato["rimosse"] |= tabelle
    lbl_gruppo.value = (f"Tolte {len(tabelle)} tabelle del gruppo <b>{dd_gruppo.value}</b>. "
                        "Si possono rimettere dalla scheda Togli entità.")
    aggiorna()


def apri_schema_iniziale(_=None):
    out_iniziale.clear_output()
    with out_iniziale:
        if stato["df_schema"] is None:
            print("Prima carica i file.")
            return
        file_html = stato["cartella"] / "schema_iniziale.html"
        G.schema_iniziale(stato["df_schema"], stato["pk"], stato["fk"], stato["gruppo_di"], file_html)
        webbrowser.open(file_html.resolve().as_uri())
        print("Salvato:", file_html)


def rimetti(_=None, tutte=False):
    scelte = set(stato["rimosse"]) if tutte else set(sel_tolte.value)
    if not scelte:
        lbl_entita.value = "Nessuna tabella selezionata."
        return
    stato["rimosse"] -= scelte
    lbl_entita.value = f"Rimesse {len(scelte)} tabelle."
    aggiorna()


def salva_file(_=None):
    out_salva.clear_output()
    with out_salva:
        if stato["R"] is None:
            print("Prima carica i file.")
            return
        originali = A.create_originali_cartella(stato["cartella"], imp.dialetto)
        for file in A.salva(stato["R"], stato["cartella"], originali, imp.dialetto):
            print("Salvato:", file)


def apri_schema(_=None):
    out_salva.clear_output()
    with out_salva:
        if stato["R"] is None:
            print("Prima carica i file.")
            return
        file_html = stato["cartella"] / "schema_collegamenti.html"
        G.schema_collegamenti(stato["R"], file_html)
        webbrowser.open(file_html.resolve().as_uri())
        print("Salvato:", file_html)


btn_analizza.on_click(carica)
btn_cerca_entita.on_click(cerca_entita)
txt_entita.observe(cerca_entita, names="value")
btn_togli.on_click(togli)
btn_rimetti.on_click(rimetti)
btn_rimetti_tutte.on_click(lambda b: rimetti(tutte=True))
btn_togli_gruppo.on_click(togli_gruppo)
btn_iniziale.on_click(apri_schema_iniziale)
dd_gruppo.observe(vista_gruppi, names="value")
btn_salva.on_click(salva_file)
btn_schema.on_click(apri_schema)
txt_cerca.observe(vista_tabelle, names="value")
dd_pk.observe(vista_pk, names="value")
for controllo in (dd_aff, chk_tipi, txt_fk):
    controllo.observe(vista_fk, names="value")

# --- impaginazione
schede = w.Tab(children=[
    w.VBox([out_panoramica, btn_iniziale, out_iniziale]),
    w.VBox([w.HBox([dd_gruppo, btn_togli_gruppo]), lbl_gruppo, out_gruppi]),
    w.VBox([txt_cerca, out_tabelle]),
    w.VBox([dd_pk, out_pk]),
    w.VBox([w.HBox([dd_aff, chk_tipi, txt_fk]), out_fk]),
    out_altri,
    w.VBox([w.HBox([txt_entita, btn_cerca_entita]), lbl_entita, sel_trovate, btn_togli,
            w.HTML("<hr><b>Tabelle tolte</b>"), sel_tolte, w.HBox([btn_rimetti, btn_rimetti_tutte])]),
    w.VBox([w.HBox([btn_salva, btn_schema]), out_salva]),
])
for i, titolo in enumerate(["Panoramica", "Gruppi", "Tabelle", "PK composte", "FK candidate", "Altri risultati",
                            "Togli entità", "Salva"]):
    schede.set_title(i, titolo)

display(w.VBox([
    w.HTML("<h3>Collegamenti tra tabelle SQL</h3>"),
    w.HBox([txt_cartella, dd_dialetto, btn_analizza]),
    lbl_stato,
    schede,
]))
carica()